In [4]:
import numpy as np
from scipy import stats

def cross_correlation_xbh(x, y, max_lag, alpha=0.05):
    """
    Calcula la correlación cruzada y los umbrales críticos de significación 
    siguiendo el método xBH de Gao et al. (2024).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    N = len(x)
    
    if len(y) != N:
        raise ValueError("Las series x e y deben tener la misma longitud N.")
    
    # 1. Estandarización (desmedido y escala)
    x_c = x - np.mean(x)
    y_c = y - np.mean(y)
    
    std_x = np.std(x_c, ddof=0)
    std_y = np.std(y_c, ddof=0)
    
    # 2. Autocorrelaciones muestrales completas
    autocorr_x = np.correlate(x_c, x_c, mode='full')[N-1:] / (N * std_x**2)
    autocorr_y = np.correlate(y_c, y_c, mode='full')[N-1:] / (N * std_y**2)
    
    lags = np.arange(-max_lag, max_lag + 1)
    n_tau = len(lags)
    
    ccf = np.zeros(n_tau)
    delta = np.zeros(n_tau)
    v_eff = np.zeros(n_tau)
    rho_c = np.zeros(n_tau)
    
    # 3. Cálculo de CCF y delta(tau) para cada lag
    for idx, tau in enumerate(lags):
        abs_tau = abs(tau)
        N_sub = N - abs_tau
        
        if tau >= 0:
            cxy = np.sum(x_c[:N_sub] * y_c[tau:]) / N
        else:
            cxy = np.sum(x_c[abs_tau:] * y_c[:N_sub]) / N
            
        ccf[idx] = cxy / (std_x * std_y)
        
        # Factor xBH: Ec. (5)
        k = np.arange(1, N_sub)
        weight = (N_sub - k) / N_sub
        sum_term = np.sum(weight * autocorr_x[k] * autocorr_y[k])
        
        denom = 1.0 + 2.0 * sum_term
        delta[idx] = 1.0 / denom if denom > 0 else 0.0
        
        # Grados de libertad efectivos: Ec. (4)
        v_eff[idx] = max(N_sub * delta[idx] - 2.0, 1.0)
        
    # 4. Ajuste por búsqueda de rango: Ecs. (6) y (7)
    delta_0 = delta[n_tau // 2]  # delta en tau = 0
    n_i = max(n_tau * delta_0, 1.0)
    alpha_c = 1.0 - (1.0 - alpha)**(1.0 / n_i)
    
    # 5. Coeficiente crítico para cada tau: Ec. (8)
    for idx, tau in enumerate(lags):
        abs_tau = abs(tau)
        t_crit = stats.t.ppf(1.0 - alpha_c / 2.0, df=v_eff[idx])
        rho_c[idx] = ((N - abs_tau) / N) * (t_crit / np.sqrt(v_eff[idx] + t_crit**2))
        
    # Identificar si la correlación observada es significativa
    is_significant = np.abs(ccf) >= rho_c
    
    return {
        'lags': lags,
        'ccf': ccf,
        'rho_critical': rho_c,
        'is_significant': is_significant,
        'v_effective': v_eff,
        'delta': delta,
        'n_independent': n_i,
        'alpha_adjusted': alpha_c
    }

# Ejemplo de uso con series sintéticas autocorrelacionadas AR(1)
if __name__ == "__main__":
    np.random.seed(42)
    N = 250
    
    # Generar dos series independientes pero con alta autocorrelación (ruido rojo)
    x = np.zeros(N)
    y = np.zeros(N)
    for t in range(1, N):
        x[t] = 0.8 * x[t-1] + np.random.normal()
        y[t] = 0.85 * y[t-1] + np.random.normal()
        
    res = cross_correlation_xbh(x, y, max_lag=20, alpha=0.05)
    
    print(f"Número de desfases independientes (n_i): {res['n_independent']:.2f}")
    print(f"Alpha ajustado por búsqueda de rango (alpha_c): {res['alpha_adjusted']:.4f}")
    print(f"Picos con correlación significativa: {np.sum(res['is_significant'])} de {len(res['lags'])}")

Número de desfases independientes (n_i): 12.21
Alpha ajustado por búsqueda de rango (alpha_c): 0.0042
Picos con correlación significativa: 0 de 41
